# 26 · Deduplicate point records against rapidfuzz, jellyfish and a geodesic

**Question.** Does Fieldwork's `deduplicate_points` widget decide that two records are one
facility the way it says it does?

The widget takes two layers of records from different sources, blocks the pairs that are too far
apart to consider, compares the remaining pairs on name and on distance, scores each comparison
as agreeing or not, sorts the pairs into three classes with two thresholds, and merges the
records a match licences -- at most one per source in any cluster.

**Why this one.** It sits on the weakest rung of Fieldwork's evidence ladder and no worked example
exercises it, so until now every claim about it rested on that repository's own tests.

**Genuinely independent, four ways.**

| part | Fieldwork | here |
| --- | --- | --- |
| name comparison | its own Jaro-Winkler | rapidfuzz (C++) and jellyfish (Rust) |
| normalisation | `\p{L}\p{N}` in JavaScript | `str.isalnum` in Python |
| distance | great circle, R = 6371.0088 km | WGS84 geodesic, pyproj |
| blocking, scoring, classes, clusters | `deduplicatePoints` | reimplemented from `declaredParameters` |

Nothing is imported from Fieldwork. Its outputs are in the fixture's `output` and are the thing
compared, never an input to the recomputation.

**The records are constructed, not downloaded.** A parity check asks whether an algorithm computes
what it claims, and that needs every branch to fire: a pair agreeing on both comparisons, a pair
agreeing on each one alone, a pair reaching neither, a pair blocking never offers, a name that is
missing, a record with no location, a same-source pair, and the cluster rule refusing a second
record from one source. Downloaded records exercise whichever of those they happen to contain.

**What this cannot establish: what the thresholds should be.** 0.88, 500 m and 50 m are proposals
in the release, not measurements. Measuring them needs records whose true identity is known.

In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["rapidfuzz", "jellyfish", "pyproj"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_26_deduplicate import run

report = run(Path.cwd() / "fixtures" / "deduplicate-facilities-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"],
      f"(names {report['nameSimilarityAgrees']}, pipeline {report['pipelineAgrees']})")
print(f"{report['records']} records, {report['placedRecords']} with a location")
print(f"{report['counts']['comparedFieldwork']} pairs compared, "
      f"{report['counts']['blockedFieldwork']} never compared")
print(f"{report['cluster']['mergedFieldwork']} records merged, "
      f"{report['cluster']['refusedCount']} matched pair(s) refused by the one-per-source rule")

## Every pair, and what each comparison said

Two binary contributions, never a blended score: a similarity of 0.87 against a threshold of 0.88
contributes nothing rather than 0.87. The class follows from the score alone.

In [ ]:
print(f"{'pair':24s} {'metres':>8s} {'geodesic':>9s}  name  near  score  class")
for row in report["pairs"]:
    pair = row["key"].replace("|", " ~ ").replace("osm:", "").replace("overture:", "")
    print(f"{pair:24s} {row['metresFieldwork']:8.1f} {row['metresReference']:9.1f} "
          f" {str(row['nameAgreesAgrees']):5s} {str(row['nearAgreesAgrees']):5s} "
          f"{str(row['scoreAgrees']):6s} {row['classFieldwork']}"
          f"{'' if row['classAgrees'] else '  <- DISAGREES'}")
print()
worst = max(report["pairs"], key=lambda r: r["metresRelative"])
print(f"largest distance disagreement: {worst['metresRelative']*100:.3f}% on "
      f"{worst['key']}, against a 1% tolerance")
print("every class agrees:", all(r["classAgrees"] for r in report["pairs"]))

## The name comparison, three ways — and one real divergence

Fieldwork, rapidfuzz and jellyfish agree exactly on every published reference pair:
`martha`/`marhta` is 0.961 and `dwayne`/`duane` is 0.840 in the literature, and all three produce
those values.

They part on one constructed case. `abcvwxyz` against `cabvwxyz` has eight matched characters and
**three** of them in a different order. Jaro halves that count: Fieldwork halves it exactly, to
1.5, and both reference libraries halve it with integer division, to 1. The third term is
therefore 6.5/8 against 7/8, and the similarity 0.9375 against 0.958333.

Two independent references agreeing with each other makes this a **convention rather than a slip
in either**, and it can only arise when the raw count is odd — which needs a cycle of odd length
among the matched characters. Every published reference pair has an even count, so no published
value can detect it. It is reached by no name pair in this fixture's two layers.

It is recorded here, with both values, rather than removed.

In [ ]:
print(f"{'a':12s} {'b':12s} {'fieldwork':>10s} {'rapidfuzz':>10s} {'jellyfish':>10s}")
for row in report["nameSimilarities"]:
    flag = "" if row["agreesWithRapidfuzz"] else "   <- diverges"
    print(f"{row['a'] or '(empty)':12s} {row['b'] or '(empty)':12s} "
          f"{row['fieldwork']:10.6f} {row['rapidfuzz']:10.6f} {row['jellyfish']:10.6f}{flag}")
print()
for d in report["divergences"]:
    print(f"{d['case']}: difference {d['difference']:.6f}")
    print(f"  {d['cause']}")

## The cluster rule, and the partner it chooses

A cluster holds at most one record from each source, so when two records from one layer both
match a single record in the other, one edge merges and the other is refused and stays reported
as a match that did not merge.

Both implementations refuse the same two edges here, because both follow the declared rule. What
the rule does with them is worth looking at: `osm-2` is the *Annex*, 2 m from Overture's record
for the main hospital and 20 m from its own layer's record for that hospital. Pairs are accepted
in order of distance, so the Annex takes the main hospital's Overture record, and the pair a
reader would choose is then refused.

That is a finding about the rule, not about either implementation of it. The widget's own
documentation is explicit that the review band is where such judgements belong — and these pairs
never reach the review band, because on name and distance alone they are matches.

In [ ]:
c = report["cluster"]
print("merged:", c["mergedFieldwork"], "(reference", str(c["mergedReference"]) + ")")
print("refused by the one-per-source rule:")
for key in c["refusedEdges"]:
    print("  ", key.replace("|", " ~ "))
print()
print("output records agree:", c["outputRecordsAgree"],
      "· every mergedFrom agrees:", c["mergedFromAgrees"])
print(f"{c['outputCountFieldwork']} records out of {report['records']} in")

### Reading the result

`pipelineAgrees` means two independent implementations of blocking, normalisation, scoring,
classification and clustering produced the same answer on every pair, every count, every output
record and every `mergedFrom`, with distances from a geodesic rather than a sphere.

`agrees` is False, and only because of the one constructed name pair above. Nothing in the
pipeline disagrees. The lab publishes the disagreement with both values rather than dropping the
case that found it.